# Week 7 — Document Question Answering System (RAG)
**Celebal Technologies — Data Science Internship | Aryan Mahanty**

This notebook documents the Week 7 RAG (Retrieval-Augmented Generation) project: an end-to-end
pipeline that answers questions grounded in a custom PDF document, using Cohere (embeddings +
generation), Pinecone (vector database), Streamlit (interface) and PyMuPDF (PDF text extraction).

Code files for this project (`app.py`, `chatbot.py`, `vectorstore.py`, `requirements.txt`) are in
the `src/` folder alongside this report.


## 1. Overview

This project implements a Retrieval-Augmented Generation (RAG) system that answers questions
grounded in a custom PDF document. The system retrieves relevant passages from the uploaded
document and uses a language model to generate an answer based only on that retrieved context,
rather than the model's general knowledge.

**Stack used:** Cohere (embeddings + generation), Pinecone (vector database), Streamlit
(interface), PyMuPDF (PDF text extraction).


## 2. System Architecture

1. **Document Ingestion** — PDF uploaded via Streamlit, text extracted using PyMuPDF (`fitz`).
2. **Chunking** — raw text split into sentence-aware chunks of a configurable character length.
3. **Embedding** — each chunk converted to a vector using Cohere's `embed-english-v3.0` model.
4. **Vector Storage** — chunk embeddings stored in a Pinecone serverless index (cosine similarity).
5. **Query Processing** — user question converted to a query embedding using the same embedding model.
6. **Retrieval** — top-k most similar chunks retrieved from Pinecone for the query.
7. **Generation** — retrieved chunks passed as grounding documents to Cohere's `command-a-03-2025`
   chat model, which streams back a grounded answer.
8. **Index reset** — vector store is cleared (`delete_all=True`) before indexing each new document,
   so answers are never mixed across documents.


## 3. System Metrics Report

| Component | Configuration |
|---|---|
| Chunking strategy | Sentence-based, chunk size ≈ 500 characters (see Section 5 for the 1000 vs 500 experiment) |
| Embedding model | Cohere `embed-english-v3.0` |
| Embedding dimensions | Matches Cohere `embed-english-v3.0` output (set dynamically from `len(embeddings[0])` at index-creation time) |
| Vector store | Pinecone — Serverless index, AWS, us-east-1, cosine similarity |
| Retrieval top_k | 10 |
| Generation model | Cohere `command-a-03-2025` |
| Interface | Streamlit (local app, PDF upload + Q&A) |


## 4. Validation Logs

Five sample questions were run against an uploaded PDF (Mu Sigma / Golgix campus placement
document) to validate retrieval and generation quality, including one edge case designed to check
for hallucination.

| Question | Result Summary | Type |
|---|---|---|
| What is the hiring process? | Explains the two-link registration requirement, 10 AM 1st June deadline, mandatory both-link registration, and includes the actual company + T&P links. | Factual |
| Say me about salary | Gives full compensation breakdown across 4 years (probation 5 LPA, yearly LPA figures), total CTC (40 Lakhs), 2-year agreement, and ₹10 Lakh recovery clause. | Factual / detailed |
| "Summarize this document in 2 lines" | Correctly condenses the whole document: role (Trainee Decision Scientist-1), eligible batch (SOA, 2027), and registration deadline. | Summary |
| What is the salary for a Python developer role? | Correctly responds: "I'm sorry, I couldn't find any information about a Python developer role." No hallucination. | Edge case (info not in document) |
| Which batch is this for? | Correctly identifies 2027 graduating batch from SOA. | Factual |

**Key result:** the edge-case question (asking about a role not present in the document) was
correctly declined rather than hallucinated, confirming the answers are grounded in retrieved
context only.


## 5. Optimization Experiment — Chunk Size

As an optimization experiment, the chunk size was reduced from 1000 to 500 characters to observe
the effect on retrieval and answer quality, using the same document and a repeated question
("Say me about salary").

| Aspect | Chunk size 1000 | Chunk size 500 |
|---|---|---|
| Chunk size | 1000 characters (sentence-based) | 500 characters (sentence-based) |
| Sample answer detail | Compensation breakdown given, but recovery clause / 2-year agreement omitted in that particular answer. | Same compensation breakdown returned with additional detail: recovery clause and 2-year agreement mentioned; formatted as a clear bullet list. |
| Observation | Larger chunks retrieved slightly broader context per chunk, but relevant details occasionally spread across multiple chunks that weren't all retrieved. | Smaller chunks increased the number of distinct chunks retrieved for the same top_k, surfacing more specific details (like the recovery clause) in the same answer. |

**Conclusion:** smaller chunk sizes increased retrieval granularity, surfacing slightly more
specific details (e.g. the recovery clause) within the same top_k. For longer or more complex
documents, this trade-off (more granular chunks vs. broader per-chunk context) would be worth
tuning further, and could be combined with hybrid (keyword + vector) search or a re-ranking step
as a next iteration.


## 6. Key Learnings

- How RAG combines retrieval (finding relevant context) with generation (producing grounded answers).
- Why retrieval quality directly bounds answer quality — chunking and embedding choices measurably
  affect what the model can see.
- Practical experience with embeddings, vector similarity search, and a managed vector database
  (Pinecone).
- The importance of resetting/scoping the vector index per document to avoid cross-document
  contamination.
- Verifying groundedness by explicitly testing an out-of-scope question to check for hallucination.


## 7. Conclusion

The system successfully implements an end-to-end document question-answering pipeline: it ingests
a custom PDF, retrieves the most relevant sections for a given query, and generates accurate,
context-aware answers — while correctly declining to answer when the requested information is not
present in the source document.
